# Chapter 13 &mdash; The Compact ID Notation $aqb$

**Concept 13 of the Chapter 13 decomposition:** *The Compact ID Notation $aqb$*

Write the tape with the state symbol inserted just before the scanned cell.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter13/Concept-Compact-ID-Notation/Concept-Compact-ID-Notation.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.Def_PDA        import *
from jove.Def_TM         import *
from jove.AnimateTM      import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


A TM's instantaneous description needs three things: the tape, the head position, and
the state. The **compact notation** packs all three into one string: write the tape
and **insert the state symbol immediately before the scanned cell**.

So $\texttt{01}q\texttt{101}$ means: tape `01101`, state $q$, head on the third cell
(the `1` right after $q$).

Two conventions make it unambiguous: state names are drawn from a set disjoint from
$\Gamma$, and trailing blanks are dropped.

Transitions become string rewrites, which is why this notation is the bridge to
**unrestricted grammars** &mdash; a TM computation *is* a derivation.

## 2. Definitions

### Compact IDs from Jove's triples

In [ ]:
Flip = md2mc('''TM
!! Flip every bit, then halt in F.  F has NO outgoing transitions, which
!! is how a Jove TM signals "halt here".
I : 0 ; 1 , R -> I
I : 1 ; 0 , R -> I
I : . ; . , S -> F
''')


def compact(cfg):
    q, head, tape, fuel = cfg
    tape = tape.rstrip('.') or '.'
    if head >= len(tape): tape = tape + '.' * (head - len(tape) + 1)
    return tape[:head] + q + tape[head:]

def compact_run(T, tape, fuel=100):
    trunc, halts = run_tm(T, tape if tape else '.', fuel, chatty=False)
    if not halts: return None
    cfg, path = halts[0]
    return [compact(c) for c in path] + [compact(cfg)]

# --- thin wrappers over Jove's TM runner --------------------------------
# run_tm(T, tape, fuel) returns (truncated-paths, haltList).  A TM HALTS
# when no transition applies, and ACCEPTS if it halts in a final state.
# So an accepting state must have NO outgoing transitions, or the machine
# will run on past it.
def tm_accepts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return any(cfg[0] in T["F"] for cfg, _ in halts)

def tm_halts(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return len(halts) > 0

def tm_tape(T, tape, fuel=200):
    trunc, halts = run_tm(T, tape if tape != '' else '.', fuel, chatty=False)
    return [cfg[2].rstrip('.') for cfg, _ in halts]

## 3. Tests

A whole computation, one line per step.

In [ ]:
for idd in compact_run(Flip, '0101'):
    print("   ", idd)

Reading one ID: tape, state, head position.

In [ ]:
ids = compact_run(Flip, '0101')
one = ids[2]
print("ID :", one)
q = [ch for ch in one if ch in Flip["Q"]][0]
pos = one.index(q)
print("  state :", q)
print("  tape  :", one.replace(q, ''))
print("  head  : cell %d (scanning %r)" % (pos, one[pos+1] if pos+1 < len(one) else '.'))
assert q in Flip["Q"]

The convention that makes it unambiguous.

In [ ]:
print("Q     :", sorted(Flip["Q"]))
print("Gamma :", sorted(Flip["Gamma"]))
print("disjoint? ", Flip["Q"] & Flip["Gamma"] == set())
assert Flip["Q"] & Flip["Gamma"] == set()
print("\nIf a state were named '0' the notation would be unreadable.")

Each step is a **string rewrite** of a bounded window.

In [ ]:
ids = compact_run(Flip, '0101')
for a, b in zip(ids, ids[1:]):
    i = next((k for k in range(min(len(a), len(b))) if a[k] != b[k]), 0)
    print("  %-10s -> %-10s   changed from position %d" % (a, b, i))
print("\nOnly a two- or three-character window changes each step.")

Which is exactly the shape of an **unrestricted grammar** production.

In [ ]:
print("TM transition   (q, a) -> (p, b, R)")
print("as a rewrite     q a   =>   b p")
print()
print("TM transition   (q, a) -> (p, b, L)")
print("as a rewrite    c q a  =>   p c b     for every tape symbol c")
print()
print("A TM computation IS a derivation in an unrestricted grammar.")
print("That is the last row of the Chomsky hierarchy, made concrete.")

## 4. Animation

The machine whose IDs you just read.

*(The `display(HTML(...))` line loads the toolbar's font-awesome icons. Keep it last in the cell &mdash; it must be there for the controls to appear.)*

In [ ]:
from jove.AnimateTM import *
AnimateTM(Flip, FuseEdges=True)
display(HTML('<link rel="stylesheet" href="//stackpath.bootstrapcdn.com/font-awesome/4.7.0/css/font-awesome.min.css"/>'))

## 5. Exercises


1. Write the compact ID for: tape `aab`, state $q_3$, head on the last cell.
2. Give the rewrite rules for an `S` (stay) move.
3. Why must trailing blanks be dropped for the notation to be canonical?

In [ ]:
# Your work for the exercises above.